# Decorator Design Pattern 

explained using the classic Coffee Shop example.

#### The Concept

The Decorator Pattern allows you to add features to an object dynamically by **"wrapping"** it in another object. Instead of creating a giant class hierarchy like CoffeeWithMilk, CoffeeWithSugar, CoffeeWithMilkAndSugar, you start with a plain Coffee and wrap it in layers.

**Analogy**: Putting on clothes. You start with your body (Component). You put on a shirt (Decorator 1). Then a jacket (Decorator 2). You are still "You", but with added layers.

## The Classic OOP Way (Java-Style)

In the Java/C++ style, we use an Abstract Base Class for the component, and a specific "Decorator" class that mimics the component while holding a reference to it.

#### THE COMPONENT INTERFACE

In [1]:
from abc import ABC, abstractmethod

class Coffee(ABC):
    @abstractmethod
    def get_cost(self) -> float:
        pass

    @abstractmethod
    def get_description(self) -> str:
        pass

#### CONCRETE COMPONENT (The Base Object)

In [2]:
class SimpleCoffee(Coffee):
    def get_cost(self) -> float:
        return 5.0

    def get_description(self) -> str:
        return "Simple Coffee"

#### THE BASE DECORATOR

In [3]:
class CoffeeDecorator(Coffee):
    """
    The 'Middleman'. It looks like a Coffee, but it holds a reference
    to another Coffee object inside it.
    """
    def __init__(self, coffee: Coffee):
        self._coffee = coffee

    def get_cost(self) -> float:
        return self._coffee.get_cost()

    def get_description(self) -> str:
        return self._coffee.get_description()

#### CONCRETE DECORATORS (The Layers)

In [4]:
class Milk(CoffeeDecorator):
    def get_cost(self) -> float:
        return self._coffee.get_cost() + 1.5

    def get_description(self) -> str:
        return self._coffee.get_description() + ", Milk"

class Sugar(CoffeeDecorator):
    def get_cost(self) -> float:
        return self._coffee.get_cost() + 0.5

    def get_description(self) -> str:
        return self._coffee.get_description() + ", Sugar"

#### CLIENT CODE

In [6]:
def main():
    # 1. Start with plain coffee
    my_coffee = SimpleCoffee()
    print(f"{my_coffee.get_description()} : ${my_coffee.get_cost()}")

    # 2. Decorate with Milk
    my_coffee = Milk(my_coffee)
    print(f"{my_coffee.get_description()} : ${my_coffee.get_cost()}")

    # 3. Decorate with Sugar
    my_coffee = Sugar(my_coffee)
    print(f"{my_coffee.get_description()} : ${my_coffee.get_cost()}")

    # 4. Decorate with Sugar
    my_coffee = Sugar(my_coffee)
    print(f"{my_coffee.get_description()} : ${my_coffee.get_cost()}")

if __name__ == "__main__":
    main()

Simple Coffee : $5.0
Simple Coffee, Milk : $6.5
Simple Coffee, Milk, Sugar : $7.0
Simple Coffee, Milk, Sugar, Sugar : $7.5


## The Pythonic Way

In Python, we can leverage `__getattr__` (Dynamic Delegation). Strictly inheriting from a `CoffeeDecorator` class is often unnecessary boilerplate. We can make a wrapper that automatically forwards any unknown method call to the wrapped object. This is much more flexible.

#### PROTOCOL (Optional Type Safety)

In [8]:
from typing import Protocol

class Beverage(Protocol):
    def cost(self) -> float: ...
    def desc(self) -> str: ...

#### CONCRETE COMPONENT

In [9]:
class Espresso:
    def cost(self) -> float: return 5.0
    def desc(self) -> str: return "Espresso"

#### PYTHONIC DECORATORS (Dynamic Wrappers)

In [12]:
from typing import Any

class Milk:
    def __init__(self, wrapped: Any):
        self._wrapped = wrapped

    def cost(self) -> float:
        return self._wrapped.cost() + 1.5

    def desc(self) -> str:
        return self._wrapped.desc() + ", Milk"
    
    # MAGIC METHOD: Dynamic Delegation
    # If the user calls a method that Milk doesn't have,
    # pass it down to the wrapped object automatically.
    def __getattr__(self, name):
        return getattr(self._wrapped, name)


class Vanilla:
    def __init__(self, wrapped: Any):
        self._wrapped = wrapped

    def cost(self) -> float:
        return self._wrapped.cost() + 2.0

    def desc(self) -> str:
        return self._wrapped.desc() + ", Vanilla"

    def __getattr__(self, name):
        return getattr(self._wrapped, name)

#### CLIENT CODE

In [13]:
def main():
    # 1. Nesting Objects directly
    # Usage: Vanilla( Milk( Espresso() ) )
    
    my_drink = Vanilla(Milk(Espresso()))

    print(f"Order: {my_drink.desc()}")
    print(f"Total: ${my_drink.cost()}")

    # 5. Accessing methods that weren't explicitly overridden?
    # Because of __getattr__, if Espresso had a method 'brew()',
    # my_drink.brew() would still work!

if __name__ == "__main__":
    main()

Order: Espresso, Milk, Vanilla
Total: $8.5


| Key Differences | Feature      | Classic OOP                                        | Pythonic                                              |
|-----------------|--------------|----------------------------------------------------|--------------------------------------------------------|
| Structure       | Approach     | Rigid inheritance (extends `CoffeeDecorator`).     | Loose composition (Duck Typing).                       |
| Boilerplate     | Code Effort  | High — must implement every method from Interface. | Low — only implement methods you want to change.       |
| Delegation      | Behavior     | Manual (`return super.method()`).                  | Automatic (`__getattr__` delegates everything else).    |


Important Note on Python's `@decorator`

Python has a built-in syntax `@decorator_name` used for functions and classes. While related (they both wrap things), `the GoF Decorator Pattern` described above is about runtime object composition, whereas Python's @decorator syntax is usually for `definition-time modification`.